# Directional Forecasting of the S&P 500 Index

*DataCamp Challenge — École Polytechnique (INF554 / MAP583)*

---

Can you predict whether the S&P 500 will close **UP** or **DOWN** tomorrow?

This notebook walks you through the data, the evaluation metric, and how to build and test a submission locally before uploading it to Codabench.


## Introduction

### The Task

This is a **binary classification** challenge: given the recent history of the S&P 500 index, predict whether the next trading day's closing price will be **strictly above** (`1`) or **at or below** (`0`) the current day's closing price.

### The Data

Each row in the dataset represents one **trading day** and contains the following raw OHLCV features:

| Column   | Description |
|----------|-------------|
| `Open`   | Opening price of the day |
| `High`   | Intraday high |
| `Low`    | Intraday low |
| `Close`  | Closing price of the day |
| `Volume` | Total trading volume |

The ingestion program wraps these rows into **sliding windows of 50 consecutive trading days**, so your model receives sequences of shape `(batch, 50, 5)`.

### Why It Matters

Predicting market direction is a canonical and challenging time-series problem — the signal-to-noise ratio is very low, and models that generalise beyond the training period are rare. The challenge rewards robust, well-regularised approaches over overfit ones.


## Exploratory Data Analysis

Let's load the raw training data and get a feel for what we're working with.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DATA_DIR = "dev_phase/input_data"

# Load raw CSV files for exploration
features = pd.read_csv(f"{DATA_DIR}/train/train_features.csv", index_col=0)
labels = pd.read_csv(f"{DATA_DIR}/train/train_labels.csv", index_col=0)

print(f"Training samples: {len(features)}")
print(f"Features: {list(features.columns)}")
print(f"\nLabel distribution:\n{labels['Target'].value_counts().rename({1: 'UP (1)', 0: 'DOWN (0)'})}")
features.head()


In [ ]:
# Plot the Close price series with UP/DOWN days coloured
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

up = labels["Target"] == 1
axes[0].plot(features.index, features["Close"], color="steelblue", linewidth=0.8)
axes[0].set_title("S&P 500 Close Price (training period)")
axes[0].set_ylabel("Close price")

axes[1].bar(features.index[up],  1, color="green", width=1, label="UP (1)")
axes[1].bar(features.index[~up], 1, color="red",   width=1, label="DOWN (0)")
axes[1].set_title("Daily direction label")
axes[1].set_ylabel("Label")
axes[1].legend(loc="upper right")
axes[1].set_xlabel("Trading day index")

plt.tight_layout()
plt.show()


In [ ]:
# Feature statistics
features.describe().round(2)


## Challenge Evaluation

Submissions are ranked by **ROC-AUC** (Area Under the ROC Curve) computed on a held-out test window.

- A **perfect** model scores **1.0**
- **Random guessing** (outputting 0.5 for every sample) scores **≈ 0.5**
- Predicting hard 0/1 labels instead of probabilities will likely score around 0.5 — always output **sigmoid probabilities**.

The key advantage of ROC-AUC is that it is **threshold-independent**: it rewards models that rank UP days above DOWN days regardless of the absolute probability values they produce.

There are two splits:
- **Public test** — visible on the leaderboard during the development phase  
- **Private test** — revealed only after the phase ends (final ranking)


## Submission Format

You must submit a **single file** named `submission.py` that exposes one function:

```python
def get_model(train_loader: torch.utils.data.DataLoader) -> torch.nn.Module:
    ...
    return model  # already in eval mode
```

**Contract:**
- `train_loader` yields `(x, y)` batches where `x` has shape `(batch, 50, 5)` and `y` has shape `(batch,)` with values in `{0, 1}`
- The returned model's `forward(x)` must accept `(batch, 50, 5)` tensors and return **probabilities in [0, 1]** of shape `(batch,)` — i.e. apply `sigmoid` inside `forward`

The cell below shows the reference LSTM baseline included in the challenge.


### Baseline: LSTM Classifier

The baseline uses a multi-layer LSTM that reads the 50-day window and outputs a direction probability from the last hidden state.
Feel free to replace this architecture entirely — a Transformer, a 1-D CNN, or even a simple MLP over flattened windows are all valid approaches.


In [ ]:
# %load solution/submission.py
import torch
import torch.nn as nn

# ── Hyper-parameters ──────────────────────────────────────────────────────────
HIDDEN_SIZE = 128
NUM_LAYERS = 3
DROPOUT = 0.1
N_EPOCHS = 3
LEARNING_RATE = 1e-4
# ─────────────────────────────────────────────────────────────────────────────


class LSTMClassifier(nn.Module):
    """Sequence-to-one LSTM: (batch, seq_len, n_features) → (batch,) probability."""

    def __init__(self, input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (batch, seq_len, hidden_size)
        last = out[:, -1, :]           # (batch, hidden_size) — last timestep
        logit = self.head(last).squeeze(-1)  # (batch,)
        return torch.sigmoid(logit)    # probability in [0, 1]


def get_model(train_loader):
    x_sample, _ = next(iter(train_loader))
    input_size = x_sample.shape[-1]   # n_features (5 for raw OHLCV)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMClassifier(input_size=input_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()

    model.train()
    for epoch in range(N_EPOCHS):
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{N_EPOCHS}  loss={total_loss / len(train_loader):.4f}")

    model.eval()
    return model


## Local Testing Pipeline

Before submitting to Codabench you can run the full ingestion + scoring pipeline locally.
This mirrors exactly what happens on the platform.


In [ ]:
import sys
sys.path.insert(0, ".")  # make sure ingestion_program/ and solution/ are importable

import torch
from ingestion_program.ingestion import get_train_dataset, get_test_dataset, evaluate_model
from scoring_program.scoring import compute_roc_auc

DATA_DIR = "dev_phase/input_data"

# ── 1. Build training DataLoader ──────────────────────────────────────────────
train_dataset = get_train_dataset(DATA_DIR)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
print(f"Training samples : {len(train_dataset)}")

# ── 2. Train the model ────────────────────────────────────────────────────────
model = get_model(train_loader)

# ── 3. Predict on the public test set ────────────────────────────────────────
test_dataset = get_test_dataset(DATA_DIR, "test")
predictions = evaluate_model(model, test_dataset)  # DataFrame with "Probability" column

# ── 4. Score against the reference labels ────────────────────────────────────
import pandas as pd
test_labels = pd.read_csv("dev_phase/reference_data/test_labels.csv")
auc = compute_roc_auc(predictions, test_labels["Target"].values)
print(f"\nROC-AUC on public test set: {auc:.4f}")


Accuracy on test set: 0.95


/home/tom/.local/miniconda/lib/python3.12/site-packages/sklearn/base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


## Submitting to Codabench

1. Write your `submission.py` — it must define `get_model(train_loader)` returning a trained `nn.Module`.
2. Create a zip containing only that file:
   ```bash
   zip submission.zip submission.py
   ```
3. Go to the challenge page on Codabench → **My Submissions** → **Upload**.

That's it — the platform will run your code, output predictions, compute the ROC-AUC, and update the leaderboard automatically.
